# LightGBM-модель для прогноза дохода

1. Легкая обработка данных
2. Сравнивнение нескольких конфигураций LightGBM с разными параметрами на временном holdout.
3. Проверка варианта без признаков с нулевой gain-важностью.
4. Лучшие LightGBM-модели объединяются в ансамбль.
6. Проведение калибровки прогнозов на validation.
7. Обучение финальных моделей.
8. Выгрузка данных

## Библиотеки, базовые настройки и открытие файлов

In [1]:
# Скачивание файла с гугл диска
!pip install -q gdown
!gdown 1nf5WwX4ZYVcZu5WEQI8AMPu2lytf6Y7g
!gdown 1Q3R7St7J8K_YefqMh56MLVzEeZ5OcJe4

Downloading...
From: https://drive.google.com/uc?id=1nf5WwX4ZYVcZu5WEQI8AMPu2lytf6Y7g
To: C:\Users\Елизавета\Downloads\train_processed_v2.csv

  0%|          | 0.00/100M [00:00<?, ?B/s]
  1%|          | 524k/100M [00:00<00:50, 1.99MB/s]
  2%|1         | 1.57M/100M [00:00<00:24, 3.97MB/s]
  2%|2         | 2.10M/100M [00:00<00:29, 3.33MB/s]
  4%|4         | 4.19M/100M [00:00<00:13, 7.36MB/s]
  5%|5         | 5.24M/100M [00:00<00:12, 7.79MB/s]
  6%|6         | 6.29M/100M [00:01<00:12, 7.78MB/s]
  7%|7         | 7.34M/100M [00:01<00:12, 7.21MB/s]
  9%|8         | 8.91M/100M [00:01<00:10, 9.11MB/s]
 10%|9         | 9.96M/100M [00:01<00:10, 8.87MB/s]
 11%|#1        | 11.5M/100M [00:01<00:09, 9.17MB/s]
 13%|#3        | 13.1M/100M [00:01<00:09, 9.65MB/s]
 14%|#4        | 14.2M/100M [00:01<00:08, 9.63MB/s]
 15%|#5        | 15.2M/100M [00:01<00:08, 9.68MB/s]
 16%|#6        | 16.3M/100M [00:02<00:08, 9.68MB/s]
 17%|#7        | 17.3M/100M [00:02<00:08, 9.75MB/s]
 18%|#8        | 18.4M/100M [00:02<

In [ ]:
# запустить один раз
!pip install -q lightgbm pyarrow scipy

In [3]:
# from pathlib import Path
import json
import warnings
import zipfile
from copy import deepcopy

import numpy as np
import pandas as pd
import lightgbm as lgb
warnings.filterwarnings("ignore")

In [6]:
RANDOM_STATE = 42

TARGET = "target"
WEIGHT = "w"
ID = "id"
DATE = "dt"

# Усреднение по двум seed в финале понадобиться.
FINAL_SEEDS = [42, 2026]

# Топ конфигураций, которые попадут в финальный LGB-ансамбль.
TOP_MODELS_FOR_BLEND = 2

# Небольшое увеличение числа деревьев при обучении на всём train.
FINAL_ITERATION_MULTIPLIER = 1.05

In [44]:
train = pd.read_csv('train_processed_v2.csv', decimal=',', sep=';')
test = pd.read_csv('test_processed_v2.csv', decimal=',', sep=';')

train[DATE] = pd.to_datetime(train[DATE], errors="raise")
test[DATE] = pd.to_datetime(test[DATE], errors="raise")

train = train.sort_values(DATE).reset_index(drop=True)
test = test.sort_values(DATE).reset_index(drop=True)

print("train:", train.shape)
print("test:", test.shape)
print("train period:", train[DATE].min(), "—", train[DATE].max())
print("test period:", test[DATE].min(), "—", test[DATE].max())

train: (76786, 233)
test: (73214, 231)
train period: 2024-01-31 00:00:00 — 2024-06-30 00:00:00
test period: 2024-07-31 00:00:00 — 2024-11-30 00:00:00


In [8]:
def wmae(y_true, y_pred, weights):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    weights = np.asarray(weights, dtype=float)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)


def weighted_median(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    order = np.argsort(values)
    values = values[order]
    weights = weights[order]

    cutoff = weights.sum() / 2
    return values[np.searchsorted(np.cumsum(weights), cutoff, side="left",)]

## Обработка данных

In [60]:
protected = [TARGET, WEIGHT, ID, DATE]

categorical_features = (
    train
    .select_dtypes(
        include=["object", "category", "string"]
    )
    .columns
    .tolist()
)

features = list(set(test.columns) - set(protected))

print("Признаков:", len(features))

print("Категориальных:", categorical_features)

Признаков: 229
Категориальных: ['gender', 'adminarea', 'city_smart_name', 'dp_ewb_last_employment_position', 'addrref']


In [61]:
# Категории синхронизируются между train и test.
# Target при этом не используется.

for feature in categorical_features:
    all_values = pd.concat(
        [
            train[feature]
            .astype("string")
            .fillna("__MISSING__"),
            test[feature]
            .astype("string")
            .fillna("__MISSING__"),
        ],
        ignore_index=True,
    )

    categories = pd.Index(all_values.unique())

    train[feature] = pd.Categorical(
        train[feature]
        .astype("string")
        .fillna("__MISSING__"),
        categories=categories,
    )
    test[feature] = pd.Categorical(
        test[feature]
        .astype("string")
        .fillna("__MISSING__"),
        categories=categories,
    )

## Валидация

In [62]:
# дополнительно проверяется предыдущий месяц.

all_dates = sorted(train[DATE].unique())

validation_dates = all_dates[-2:]


folds = []

for validation_date in validation_dates:
    fit_index = train.index[train[DATE] < validation_date].to_numpy()

    validation_index = train.index[train[DATE] == validation_date].to_numpy()

    if len(fit_index) and len(validation_index):
        folds.append(
            (
                fit_index,
                validation_index,
                validation_date,
            )
        )

for fold_number, (
    fit_index,
    validation_index,
    validation_date,
) in enumerate(folds, 1):
    print(
        f"Fold {fold_number}: "
        f"fit={len(fit_index):,}; "
        f"valid={len(validation_index):,}; "
        f"date={pd.Timestamp(validation_date).date()}"
    )

Fold 1: fit=44,379; valid=16,193; date=2024-05-31
Fold 2: fit=60,572; valid=16,214; date=2024-06-30


## Параметры

In [23]:
COMMON_PARAMS = {
    "objective": "regression_l1",
    "metric": "l1",
    "learning_rate": 0.03,
    "n_estimators": 2400,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": -1,
    "force_col_wise": True,
    "deterministic": True,
    "subsample_freq": 1,
    "max_bin": 255,
}

PARAMETER_CANDIDATES = {
    "fixed_original": {
        "num_leaves": 90,
        "min_child_samples": 59,
        "subsample": 0.80,
        "colsample_bytree": 0.81,
        "reg_alpha": 0.015,
        "reg_lambda": 0.28,
        "cat_smooth": 20,
        "cat_l2": 10,
    },
    "regularized_63": {
        "num_leaves": 63,
        "min_child_samples": 100,
        "subsample": 0.86,
        "colsample_bytree": 0.90,
        "reg_alpha": 0.05,
        "reg_lambda": 1.50,
        "cat_smooth": 25,
        "cat_l2": 15,
    },
    "wide_127": {
        "num_leaves": 127,
        "min_child_samples": 80,
        "subsample": 0.82,
        "colsample_bytree": 0.90,
        "reg_alpha": 0.02,
        "reg_lambda": 2.00,
        "cat_smooth": 20,
        "cat_l2": 10,
    },
    "extra_trees_127": {
        "num_leaves": 127,
        "min_child_samples": 70,
        "subsample": 0.86,
        "colsample_bytree": 0.90,
        "reg_alpha": 0.02,
        "reg_lambda": 1.00,
        "extra_trees": True,
        "cat_smooth": 20,
        "cat_l2": 10,
    },
    "strong_regularization": {
        "num_leaves": 47,
        "min_child_samples": 140,
        "subsample": 0.90,
        "colsample_bytree": 0.92,
        "reg_alpha": 0.15,
        "reg_lambda": 3.00,
        "cat_smooth": 35,
        "cat_l2": 20,
    },
    }

print("Конфигураций:", list(PARAMETER_CANDIDATES))

Конфигураций: ['fixed_original', 'regularized_63', 'wide_127', 'extra_trees_127', 'strong_regularization']


## Функция поиска наилучших параметров

In [26]:
def train_configuration(
    configuration_name,
    configuration_params,
    selected_features,
):
    fold_scores = []
    fold_iterations = []
    latest_validation_prediction = None
    latest_validation_index = None
    latest_model = None

    for fold_number, (
        fit_index,
        validation_index,
        validation_date,
    ) in enumerate(folds, 1):
        model_params = deepcopy(COMMON_PARAMS)
        model_params.update(configuration_params)

        model = lgb.LGBMRegressor(
            **model_params
        )

        X_fit = train.loc[
            fit_index,
            selected_features
        ]
        y_fit = train.loc[
            fit_index,
            TARGET
        ]
        w_fit = train.loc[
            fit_index,
            WEIGHT
        ]

        X_validation = train.loc[
            validation_index,
            selected_features
        ]
        y_validation = train.loc[
            validation_index,
            TARGET
        ]
        w_validation = train.loc[
            validation_index,
            WEIGHT
        ]

        selected_categorical = [
            feature
            for feature in categorical_features
            if feature in selected_features
        ]

        model.fit(
            X_fit,
            y_fit,
            sample_weight=w_fit,
            eval_set=[
                (
                    X_validation,
                    y_validation,
                )
            ],
            eval_sample_weight=[
                w_validation
            ],
            categorical_feature=selected_categorical,
            callbacks=[
                lgb.early_stopping(
                    120,
                    verbose=False,
                ),
                lgb.log_evaluation(400),
            ],
        )

        prediction = np.maximum(
            model.predict(X_validation),
            0,
        )

        score = wmae(
            y_validation,
            prediction,
            w_validation,
        )

        fold_scores.append(score)
        fold_iterations.append(
            int(model.best_iteration_)
        )

        latest_validation_prediction = prediction
        latest_validation_index = validation_index
        latest_model = model

        print(
            f"{configuration_name} | "
            f"fold={fold_number} | "
            f"WMAE={score:,.4f} | "
            f"iteration={model.best_iteration_}"
        )

    # Последний фолд имеет наибольший вес, так как он ближе к test. примерно сделаем 0.35 к 0.65
    if len(fold_scores) == 1:
        selection_score = fold_scores[0]
    else:
        selection_score = (
            0.35 * fold_scores[-2]
            + 0.65 * fold_scores[-1]
        )

    return {
        "configuration": configuration_name,
        "selection_wmae": float(selection_score),
        "mean_wmae": float(np.mean(fold_scores)),
        "last_fold_wmae": float(fold_scores[-1]),
        "best_iteration": int(
            np.median(fold_iterations)
        ),
        "latest_validation_prediction": (
            latest_validation_prediction
        ),
        "latest_validation_index": (
            latest_validation_index
        ),
        "latest_model": latest_model,
        "selected_features": selected_features,
        "params": configuration_params,
    }

In [63]:
configuration_results = []

for configuration_name, configuration_params in (
    PARAMETER_CANDIDATES.items()
):
    configuration_results.append(
        train_configuration(
            configuration_name,
            configuration_params,
            features,
        )
    )

results_table = pd.DataFrame([
    {
        "configuration": result["configuration"],
        "selection_wmae": result["selection_wmae"],
        "mean_wmae": result["mean_wmae"],
        "last_fold_wmae": result["last_fold_wmae"],
        "best_iteration": result["best_iteration"],
        "feature_count": len(result["selected_features"]),
    }
    for result in configuration_results
]).sort_values("selection_wmae")

display(results_table)

best_result = min(
    configuration_results,
    key=lambda item: item["selection_wmae"],
)

print("Лучшая конфигурация:", best_result["configuration"])
print("Selection WMAE:", best_result["selection_wmae"])

[400]	valid_0's l1: 60038.9
[800]	valid_0's l1: 59764
[1200]	valid_0's l1: 59403.9
fixed_original | fold=1 | WMAE=59,381.5098 | iteration=1121
[400]	valid_0's l1: 61713
[800]	valid_0's l1: 60953.6
fixed_original | fold=2 | WMAE=60,933.6342 | iteration=943
[400]	valid_0's l1: 60376.1
[800]	valid_0's l1: 59950
[1200]	valid_0's l1: 59843
[1600]	valid_0's l1: 59782.1
[2000]	valid_0's l1: 59717.8
[2400]	valid_0's l1: 59679
regularized_63 | fold=1 | WMAE=59,677.7951 | iteration=2379
[400]	valid_0's l1: 61967.1
[800]	valid_0's l1: 61201.2
[1200]	valid_0's l1: 61034.2
[1600]	valid_0's l1: 60567.9
regularized_63 | fold=2 | WMAE=60,564.9982 | iteration=1584
[400]	valid_0's l1: 60299.8
[800]	valid_0's l1: 59873.3
[1200]	valid_0's l1: 59799
[1600]	valid_0's l1: 59721.7
wide_127 | fold=1 | WMAE=59,705.6139 | iteration=1789
[400]	valid_0's l1: 61085.2
[800]	valid_0's l1: 60387.1
wide_127 | fold=2 | WMAE=60,363.1815 | iteration=945
[400]	valid_0's l1: 61068.2
[800]	valid_0's l1: 60338.4
[1200]	valid_

,configuration,selection_wmae,mean_wmae,last_fold_wmae,best_iteration,feature_count
2,wide_127,60133.032857,60034.397718,60363.181515,1367,229
1,regularized_63,60254.477093,60121.396639,60564.998154,1981,229
0,fixed_original,60390.390689,60157.572028,60933.634231,1032,229
4,strong_regularization,60544.240788,60437.734149,60792.756280,2391,229
3,extra_trees_127,60550.730112,60408.428300,60882.767673,2393,229


Лучшая конфигурация: wide_127
Selection WMAE: 60133.032857375845


## Попытка улучшить скор за счет важности признаков

In [65]:
# убираем признаки с нулевой gain-важностью в лучшей модели.


pruning_result = None

best_model = best_result["latest_model"]

importance = pd.DataFrame({
    "feature": best_result["selected_features"],
    "gain": best_model.booster_.feature_importance(
        importance_type="gain"
    ),
    "split": best_model.booster_.feature_importance(
        importance_type="split"
    ),
}).sort_values("gain", ascending=False)

zero_gain_features = importance.loc[
    (importance["gain"] == 0)
    & (importance["split"] == 0),
    "feature",
].tolist()

pruned_features = [
    feature
    for feature in features
    if feature not in zero_gain_features
]

print("Zero-gain признаков:", len(zero_gain_features))

if (
    len(zero_gain_features) >= 3
    and len(pruned_features) >= 20
):
    pruning_result = train_configuration(
        (
            best_result["configuration"]
            + "_zero_gain_pruned"
        ),
        best_result["params"],
        pruned_features,
    )

    configuration_results.append(
        pruning_result
    )

    print(
        "Pruned WMAE:",
        pruning_result["selection_wmae"],
    )


Zero-gain признаков: 3
[400]	valid_0's l1: 60189.7
[800]	valid_0's l1: 59966.8
[1200]	valid_0's l1: 59897
wide_127_zero_gain_pruned | fold=1 | WMAE=59,780.7133 | iteration=1346
[400]	valid_0's l1: 61373.8
[800]	valid_0's l1: 61144.6
[1200]	valid_0's l1: 60471.5
[1600]	valid_0's l1: 60414.1
wide_127_zero_gain_pruned | fold=2 | WMAE=60,409.8634 | iteration=1550
Pruned WMAE: 60189.66087063511


Скор ухудшился, значит не удаляем признаки

## Построение ансамбля

In [93]:
# Выбираем лучшие модели для LightGBM-ансамбля.

configuration_results = sorted(
    configuration_results,
    key=lambda item: item["selection_wmae"],
)

ensemble_candidates = (configuration_results[:TOP_MODELS_FOR_BLEND])

latest_validation_index = (
    ensemble_candidates[0][
        "latest_validation_index"
    ]
)

y_latest = train.loc[
    latest_validation_index,
    TARGET
].to_numpy()

w_latest = train.loc[
    latest_validation_index,
    WEIGHT
].to_numpy()

# склеиваем предсказания лучших моделей на валидации
prediction_matrix = np.column_stack([
    candidate[
        "latest_validation_prediction"
    ]
    for candidate in ensemble_candidates
])

print(
    "Ансамбль из:",
    [
        candidate["configuration"]
        for candidate in ensemble_candidates
    ],
)

Ансамбль из: ['wide_127', 'wide_127_zero_gain_pruned']


In [94]:
#  оптимизация поиска весов для смешивания (блендинга) предсказаний моделей. 
# т.е. с какими коэффициентами нужно сложить предсказания двух лучших моделей, чтобы получить минимальную ошибку WMAE на валидационной выборке.


blend_grid = []

for first_weight in np.linspace(
    0,
    1,
    81,
):
    weights = np.array([
        first_weight,
        1 - first_weight,
    ])

    blended_prediction = (
        prediction_matrix @ weights
    )

    blend_grid.append({
        "first_weight": float(first_weight),
        "second_weight": float(1 - first_weight),
        "wmae": wmae(
            y_latest,
            blended_prediction,
            w_latest,
        ),
    })

blend_grid = (
    pd.DataFrame(blend_grid)
    .sort_values("wmae")
    .reset_index(drop=True)
)

display(blend_grid.head(10))

blend_weights = np.array([
    blend_grid.loc[0, "first_weight"],
    blend_grid.loc[0, "second_weight"],
])

raw_blend_prediction = (
    prediction_matrix @ blend_weights
)

raw_blend_score = wmae(
    y_latest,
    raw_blend_prediction,
    w_latest,
)

print("Blend weights:", blend_weights)
print("Raw blend WMAE:", raw_blend_score)

,first_weight,second_weight,wmae
0,0.5375,0.4625,60145.024664
1,0.5500,0.4500,60145.084381
2,0.5250,0.4750,60145.195933
3,0.5625,0.4375,60145.441661
4,0.5125,0.4875,60145.537796
5,0.5000,0.5000,60145.986563
6,0.5750,0.4250,60146.262861
7,0.4875,0.5125,60146.791312
8,0.4750,0.5250,60147.797446
9,0.5875,0.4125,60148.113210


Blend weights: [0.5375 0.4625]
Raw blend WMAE: 60145.02466387211


## Посткалибровка

In [95]:
# Осторожная посткалибровка: итоговое_предсказание = (Сырое_предсказание * Масштаб) + Сдвиг
# Калибровка используется только при улучшении не менее 0.1%.

MIN_CALIBRATION_IMPROVEMENT = 0.001

calibration_rows = []

for scale in np.linspace(
    0.98,
    1.02,
    41,
):
    scaled_prediction = (
        raw_blend_prediction * scale
    )

    shift = weighted_median(
        y_latest - scaled_prediction,
        w_latest,
    )

    calibrated_prediction = (
        scaled_prediction + shift
    )

    score = wmae(
        y_latest,
        calibrated_prediction,
        w_latest,
    )

    calibration_rows.append({
        "scale": float(scale),
        "shift": float(shift),
        "wmae": float(score),
    })

calibration_table = (
    pd.DataFrame(calibration_rows)
    .sort_values("wmae")
    .reset_index(drop=True)
)

display(calibration_table.head(10))

best_calibration = calibration_table.iloc[0]

calibration_improvement = (
    raw_blend_score
    - best_calibration["wmae"]
) / raw_blend_score

if calibration_improvement >= MIN_CALIBRATION_IMPROVEMENT:
    FINAL_SCALE = float(
        best_calibration["scale"]
    )
    FINAL_SHIFT = float(
        best_calibration["shift"]
    )
    calibrated_validation_score = float(
        best_calibration["wmae"]
    )
    USE_CALIBRATION = True
else:
    FINAL_SCALE = 1.0
    FINAL_SHIFT = 0.0
    calibrated_validation_score = float(
        raw_blend_score
    )
    USE_CALIBRATION = False

print("Use calibration:", USE_CALIBRATION)
print("Scale:", FINAL_SCALE)
print("Shift:", FINAL_SHIFT)
print(
    "Final validation WMAE:",
    calibrated_validation_score,
)

,scale,shift,wmae
0,1.020,-1252.931610,60050.143097
1,1.019,-1196.468557,60052.822805
2,1.018,-1101.483613,60055.849825
3,1.017,-1032.118570,60059.129201
4,1.016,-976.760189,60062.496796
5,1.015,-889.659938,60066.213875
6,1.014,-813.214855,60070.204565
7,1.013,-760.399826,60074.280230
8,1.012,-679.568356,60078.444159
9,1.011,-642.013506,60082.863445


Use calibration: True
Scale: 1.02
Shift: -1252.9316099063435
Final validation WMAE: 60050.14309713621


## Финальное обучение

In [97]:
# Для каждой выбранной конфигурации обучаются два seed.
# Разные seed создают небольшое разнообразие благодаря включённому bagging.

test_prediction_by_model = []

for candidate_number, candidate in enumerate(
    ensemble_candidates
):
    selected_features = candidate[
        "selected_features"
    ]

    selected_categorical = [
        feature
        for feature in categorical_features
        if feature in selected_features
    ]

    model_seed_predictions = []

    final_iterations = max(
        100,
        int(
            round(
                candidate["best_iteration"]
                * FINAL_ITERATION_MULTIPLIER
            )
        ),
    )

    for seed in FINAL_SEEDS:
        final_params = deepcopy(
            COMMON_PARAMS
        )
        final_params.update(
            candidate["params"]
        )

        final_params["random_state"] = seed
        final_params["n_estimators"] = (
            final_iterations
        )

        final_model = lgb.LGBMRegressor(
            **final_params
        )

        final_model.fit(
            train[selected_features],
            train[TARGET],
            sample_weight=train[WEIGHT],
            categorical_feature=selected_categorical,
        )

        model_prediction = np.maximum(
            final_model.predict(
                test[selected_features]
            ),
            0,
        )

        model_seed_predictions.append(
            model_prediction
        )

        print(
            f"Final {candidate['configuration']} | "
            f"seed={seed} | "
            f"iterations={final_iterations}"
        )

    averaged_model_prediction = np.mean(
        model_seed_predictions,
        axis=0,
    )

    test_prediction_by_model.append(
        averaged_model_prediction
    )

test_prediction_matrix = np.column_stack(
    test_prediction_by_model
)

raw_test_prediction = (
    test_prediction_matrix @ blend_weights
)

final_test_prediction = (
    raw_test_prediction * FINAL_SCALE
    + FINAL_SHIFT
)

final_test_prediction = np.maximum(
    final_test_prediction,
    0,
)

Final wide_127 | seed=42 | iterations=1435
Final wide_127 | seed=2026 | iterations=1435
Final wide_127_zero_gain_pruned | seed=42 | iterations=1520
Final wide_127_zero_gain_pruned | seed=2026 | iterations=1520


## Сохранение результатов

In [99]:
# Сохранение результатов

def save_submission(prediction, filename):
    prediction = np.asarray(
        prediction,
        dtype=float,
    )

    submission = pd.DataFrame({
        ID: test[ID].values,
        "predict": prediction,
    })

    submission = submission.sort_values("id").reset_index(drop=True)
    submission.to_csv(filename, sep=";", decimal=",", index=False)

    return True


submission_ = save_submission(
    final_test_prediction,
    "submission_lgb_improved.csv",
)

print("Файл скачан")

Файл скачан
